# 🚗 Automobile Workshop Customer Support Chatbot using RAG

## 📌 Project Overview

This project implements a **command-line customer support chatbot** for an automobile workshop using **Retrieval-Augmented Generation (RAG)**.

Instead of relying solely on the Large Language Model (LLM), the chatbot first retrieves relevant information from a local knowledge base stored in **ChromaDB**. The retrieved information is then provided as context to the **Cerebras LLM**, enabling it to generate accurate and context-aware responses.

---

# 📚 Step 1: Import Required Libraries

## Objective

Import all the required Python libraries for:

- Data manipulation
- Environment management
- Embedding generation
- Vector database operations
- LLM interaction

This forms the foundation of the RAG pipeline.

---

In [3]:
import os
import json
from typing import List, Dict, Any

# Data manipulation and analysis
import pandas as pd

# Vector database for semantic search
import chromadb
from chromadb.utils import embedding_functions

# Open-source embedding model library
from sentence_transformers import SentenceTransformer


print("All required libraries imported successfully.")

All required libraries imported successfully.


# 📂 Step 2: Load the Dataset

## Objective

Load the automobile workshop knowledge base into a Pandas DataFrame.

Each row in the dataset represents a single knowledge document that can later be retrieved during semantic search.

Typical information includes:

- Vehicle Name
- Problem
- Solution
- Category
- Keywords

---

In [5]:
# Define the path to your locally saved CSV dataset file
# Modify 'automobile_workshop_dataset.csv' if your filename is different
dataset_path = "./customer_support_dataset.csv"

# Load the comprehensive dataset into our main Pandas DataFrame
df = pd.read_csv(dataset_path)

# Verify successful loading by checking the shape and properties
print(f"Total Rows (Knowledge Documents): {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("\nFirst 3 rows of the loaded dataset:")
print(df.head(3))

Total Rows (Knowledge Documents): 2000
Total Columns: 9

First 3 rows of the loaded dataset:
    id         category       vehicle  \
0  101              FAQ  Maruti Swift   
1  102           Repair   Maruti Eeco   
2  103  Troubleshooting  Maruti Dzire   

                                     problem  \
0  What is the recommended service interval?   
1                    Fuel efficiency dropped   
2                   Exhaust smoke on startup   

                                            solution  \
0  Most vehicles recommend service every 10,000 k...   
1  Inspect air filter and replace if clogged. Che...   
2  Blue smoke suggests worn valve seals or piston...   

                                 keywords severity            source  \
0   service,interval,maintenance,schedule     High               FAQ   
1            fuel,mileage,filter,injector     High  Service Bulletin   
2  exhaust,smoke,valve seals,piston rings      Low   Workshop Manual   

  last_updated  
0   2024-09-07  
1

# 🔍 Step 3: Explore the Dataset

## Objective

Before preprocessing, inspect the dataset to understand its structure.

This includes checking:

- Dataset size
- Missing values
- Duplicate records
- Column names
- Data types

Understanding the data helps identify issues before building the vector database.

---

In [6]:
def explore_workshop_dataset(dataframe: pd.DataFrame) -> None:
    """
    Performs a baseline exploration of the workshop dataset 
    to identify missing values, duplicates, and data types.
    """
    print("=== Dataset Structural Summary ===")
    print(f"Shape: {dataframe.shape[0]} rows, {dataframe.shape[1]} columns\n")
    
    print("=== Column Names & Data Types ===")
    print(dataframe.dtypes)
    print("\n")
    
    print("=== Missing Values Per Column ===")
    missing_counts = dataframe.isnull().sum()
    print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values found.")
    print("\n")
    
    print("=== Duplicate Records ===")
    duplicate_count = dataframe.duplicated(subset=['id']).sum()
    print(f"Number of duplicate IDs: {duplicate_count}")
    
    # Check for empty strings or whitespace-only strings in critical columns
    print("\n=== Critical Text Column Validation ===")
    for col in ['problem', 'solution']:
        if col in dataframe.columns:
            empty_strings = (dataframe[col].astype(str).str.strip() == "").sum()
            print(f"Blank/Whitespace entries in '{col}': {empty_strings}")

# Execute the exploration function on our loaded dataframe
explore_workshop_dataset(df)

=== Dataset Structural Summary ===
Shape: 2000 rows, 9 columns

=== Column Names & Data Types ===
id               int64
category        object
vehicle         object
problem         object
solution        object
keywords        object
severity        object
source          object
last_updated    object
dtype: object


=== Missing Values Per Column ===
No missing values found.


=== Duplicate Records ===
Number of duplicate IDs: 0

=== Critical Text Column Validation ===
Blank/Whitespace entries in 'problem': 0
Blank/Whitespace entries in 'solution': 0


# 🧹 Step 4: Data Cleaning

## Objective

Prepare the dataset for embedding generation.

Cleaning steps include:

- Removing duplicate records
- Handling missing values
- Standardizing text formatting
- Removing unnecessary whitespace
- Ensuring consistent capitalization

A clean dataset improves retrieval quality.

---

In [7]:
def clean_workshop_dataset(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans the workshop dataset by removing duplicates, handling missing data,
    and standardizing text column whitespaces.
    """
    # Create a copy to prevent SettingWithCopyWarning
    cleaned_df = dataframe.copy()
    
    # 1. Remove duplicate records based on unique 'id'
    cleaned_df = cleaned_df.drop_duplicates(subset=['id'], keep='first')
    
    # 2. Handle missing values for critical and optional text columns
    critical_cols = ['problem', 'solution']
    optional_cols = ['category', 'vehicle', 'keywords', 'severity']
    
    # Drop rows where critical context is missing entirely
    cleaned_df = cleaned_df.dropna(subset=critical_cols)
    
    # Fill optional missing metadata with standard placeholders
    for col in optional_cols:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].fillna(f"Unknown_{col.capitalize()}")
            
    # 3. Standardize text formatting: strip whitespaces
    # We preserve original casing for problem/solution to maintain natural reading context for the LLM
    text_columns = ['category', 'vehicle', 'problem', 'solution', 'keywords', 'severity']
    for col in text_columns:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].astype(str).str.strip()
            
    return cleaned_df

# Execute cleaning phase and replace our working dataframe
df = clean_workshop_dataset(df)
print(f"Data cleaning complete. Retained rows: {len(df)}")

Data cleaning complete. Retained rows: 2000


# 📝 Step 5: Create Searchable Documents

## Objective

Instead of embedding multiple columns separately, combine relevant columns into a single document.

Example structure:

Vehicle

Problem

Solution

Keywords

This combined document becomes the semantic representation stored inside the vector database.

---

In [8]:
def construct_searchable_text(row: pd.Series) -> str:
    """
    Combines key columns into a structured, single document string
    optimized for semantic embedding and retrieval.
    """
    document = (
        f"Vehicle: {row['vehicle']}\n"
        f"Category: {row['category']}\n"
        f"Problem: {row['problem']}\n"
        f"Solution: {row['solution']}\n"
        f"Keywords: {row['keywords']}"
    )
    return document

# Create the new 'searchable_document' column by applying the structure across rows
df['searchable_document'] = df.apply(construct_searchable_text, axis=1)

# Display a quick preview of the first searchable document block
print("Searchable documents constructed successfully.\n")
print("=== Sample Constructed Document ===")
print(df['searchable_document'].iloc[0])

Searchable documents constructed successfully.

=== Sample Constructed Document ===
Vehicle: Maruti Swift
Category: FAQ
Problem: What is the recommended service interval?
Solution: Most vehicles recommend service every 10,000 km or 12 months, whichever comes first. Refer to the owner's manual for model-specific intervals and check severe usage schedules if driven in dusty or stop-and-go conditions.
Keywords: service,interval,maintenance,schedule


# 🧠 Step 6: Generate Embeddings

## Objective

Convert each document into a dense numerical vector using an embedding model.

Embeddings capture the semantic meaning of text, allowing the chatbot to retrieve relevant information even when the wording differs.

For example:

User Query

> My car gets too hot.

Stored Document

> Engine overheating caused by low coolant.

Although the wording differs, both texts have similar semantic meaning.

---

In [11]:
# Initialize the SentenceTransformer model locally
# 'all-MiniLM-L6-v2' is a lightweight, industry-standard model optimized for semantic search
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("Generating semantic embeddings for the knowledge base...")

# Convert our list of searchable documents into their dense vector representations
# We extract the series as a list of strings for batch processing
documents_list = df['searchable_document'].tolist()
embeddings = embedding_model.encode(documents_list, show_progress_bar=True)

# Assign the generated embeddings array back to our DataFrame as a list of vectors
df['embeddings'] = list(embeddings)

print(f"\nSuccessfully generated {len(df)} embeddings.")
print(f"Vector dimensionality: {len(df['embeddings'].iloc[0])} dimensions.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7824.91it/s]


Generating semantic embeddings for the knowledge base...


Batches: 100%|██████████| 63/63 [00:04<00:00, 14.99it/s]


Successfully generated 2000 embeddings.
Vector dimensionality: 384 dimensions.


# 🗄️ Step 7: Initialize ChromaDB

## Objective

Create a local vector database to store document embeddings.

Each stored record contains:

- Document ID
- Document Text
- Metadata
- Embedding Vector

ChromaDB enables fast semantic similarity search.

---

In [13]:
from chromadb.api.types import EmbeddingFunction, Documents

# Define the local directory path where the vector database files will reside on disk
chroma_db_dir = "automobile_workshop_vector_db"

# Instantiate a persistent disk client
chroma_client = chromadb.PersistentClient(path=chroma_db_dir)

# Define a lightweight custom embedding wrapper matching Chroma's protocol requirements
class LocalCustomEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, model):
        self.model = model
        
    def __call__(self, input: Documents) -> List[List[float]]:
        # Encodes incoming text queries natively using our shared notebook model object
        # The protocol signature uses 'input' as the parameter name
        vectors = self.model.encode(list(input), convert_to_numpy=False)
        return [list(v) for v in vectors]
    
    def name(self) -> str:
        return "LocalCustomEmbeddingFunction"

# Initialize our custom function using the pre-instantiated embedding_model from Step 6
custom_embedding_fn = LocalCustomEmbeddingFunction(embedding_model)

# Define a unique collection name for our automobile workshop documentation
collection_name = "workshop_knowledge_base"

# Create a fresh collection or fetch the existing structure if already saved
workshop_collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=custom_embedding_fn,
    metadata={"hnsw:space": "cosine"} # Explicitly configure cosine similarity metrics
)

print(f"ChromaDB initialized successfully at local directory: '{chroma_db_dir}/'")
print(f"Collection '{collection_name}' is ready to accept knowledge base index data.")

ChromaDB initialized successfully at local directory: 'automobile_workshop_vector_db/'
Collection 'workshop_knowledge_base' is ready to accept knowledge base index data.


# 💾 Step 8: Store Documents in ChromaDB

## Objective

Generate embeddings for every document and store them inside ChromaDB.

Each document is stored together with its metadata to improve retrieval accuracy.

At this stage, the knowledge base becomes searchable.

---

In [14]:
# Convert dataset columns into a list of dictionaries for explicit metadata tracking
metadata_list = []
for _, row in df.iterrows():
    metadata_list.append({
        "vehicle": str(row['vehicle']),
        "category": str(row['category']),
        "keywords": str(row['keywords']),
        "severity": str(row['severity']),
        "source": str(row['source']),
        "last_updated": str(row['last_updated']),
        "problem": str(row['problem'])
    })

# Convert identifiers and textual blocks into native lists of strings
ids_list = df['id'].astype(str).tolist()
documents_list = df['searchable_document'].tolist()
embeddings_list = df['embeddings'].tolist()

print(f"Ingesting {len(ids_list)} documents into ChromaDB storage...")

# Bulk insert indices directly into our initialized collection configuration
# We feed the pre-calculated embedding vectors directly to avoid redundant calculations
workshop_collection.add(
    ids=ids_list,
    embeddings=embeddings_list,
    documents=documents_list,
    metadatas=metadata_list
)

print("Ingestion complete. Knowledge base successfully synchronized to disk.")
print(f"Current collection item count: {workshop_collection.count()}")

Ingesting 2000 documents into ChromaDB storage...
Ingestion complete. Knowledge base successfully synchronized to disk.
Current collection item count: 2000


# 🔎 Step 9: Implement Semantic Retrieval

## Objective

Create a retrieval function that searches the vector database.

Retrieval Process:

1. Convert the user query into an embedding.
2. Compare it against stored document embeddings.
3. Retrieve the Top-K most similar documents.
4. Return the retrieved context.

This is the retrieval component of the RAG pipeline.

In [ ]:
# Re-define our custom embedding function class with explicit PyTorch tensor casting
class LocalCustomEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, model):
        self.model = model
        
    def __call__(self, input: Documents) -> List[List[float]]:
        # Generate the text embeddings using our initialized model
        vectors = self.model.encode(list(input), convert_to_tensor=True)
        
        # If the output is a PyTorch tensor, extract it to CPU space and drop it to primitive lists
        if hasattr(vectors, "cpu"):
            return vectors.cpu().tolist()
            
        return [list(v) for v in vectors]
    
    def name(self) -> str:
        return "LocalCustomEmbeddingFunction"

# 1. Re-initialize our updated wrapper function
custom_embedding_fn = LocalCustomEmbeddingFunction(embedding_model)

# 2. Bind the updated wrapper class directly back to your active Chroma collection
workshop_collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=custom_embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

print("Custom embedding function updated to handle tensor device outputs successfully.")

def retrieve_relevant_context(user_query: str, top_k: int = 3) -> List[str]:
    """
    Embeds the user's query and retrieves the top-k most semantically similar
    documents from the ChromaDB collection.
    """
    results = workshop_collection.query(
        query_texts=[user_query],
        n_results=top_k
    )
    # results['documents'] is a list of lists (one sublist per query text)
    retrieved_docs = results['documents'][0] if results['documents'] else []
    return retrieved_docs

print("retrieve_relevant_context() defined successfully.")

Custom embedding function updated to handle tensor device outputs successfully.


# 🤖 Step 10: Initialize the Cerebras LLM

## Objective

Configure the Cerebras API client for response generation.

The LLM does not answer directly from its own knowledge.

Instead, it uses the retrieved context supplied by the retrieval step to produce grounded responses.

---

In [36]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from the .env file
load_dotenv()

# Read the Cerebras API key
api_key = os.getenv("CEREBRAS_API_KEY")

if not api_key:
    raise ValueError(
        "CRITICAL ERROR: CEREBRAS_API_KEY not found.\n"
        "Ensure your .env file contains:\n"
        "CEREBRAS_API_KEY=your_api_key"
    )

# Initialize the Cerebras client
client = OpenAI(
    api_key=api_key,
    base_url="https://api.cerebras.ai/v1"
)

MODEL = "gpt-oss-120b"

print("✅ Cerebras LLM client initialized successfully.")
print(f"Using model: {MODEL}")

✅ Cerebras LLM client initialized successfully.
Using model: gpt-oss-120b


# 📝 Step 11: Build the Prompt Template

## Objective

Construct a prompt that combines:

- System Instructions
- Retrieved Context
- User Question

This guides the LLM to answer using only the retrieved workshop knowledge.

---

In [37]:
def build_rag_prompt(user_query: str, retrieved_contexts: List[str]) -> List[Dict[str, str]]:
    """
    Combines system guidelines, matching knowledge base snippets, 
    and the user's inquiry into a structured prompt payload for the LLM.
    """
    # Combine the list of retrieved context blocks into a single string section
    joined_context = "\n\n---\n\n".join(retrieved_contexts) if retrieved_contexts else "No specific documents found."
    
    # Define corporate boundaries and personality rules for the assistant
    system_instruction = (
        "You are an expert Automobile Workshop Customer Support Assistant.\n"
        "Your goal is to provide accurate, professional, and clear answers to customer inquiries.\n\n"
        "CRITICAL RULES:\n"
        "1. Rely ONLY on the information provided in the 'RETIREVED WORKSHOP CONTEXT' block below.\n"
        "2. Do NOT use outside general knowledge or guess answers.\n"
        "3. If the provided context does not contain the answer to the user's question, respond exactly with:\n"
        "   'I'm sorry, I couldn't find relevant information in our workshop database. Let me connect you with a service advisor.'\n"
        "4. Keep your answers concise, clear, and actionable."
    )
    
    # Structure the user prompt with explicit separation markers
    user_content = (
        f"RETIREVED WORKSHOP CONTEXT:\n"
        f"{joined_context}\n"
        f"---------------------------\n\n"
        f"CUSTOMER INQUIRY: {user_query}\n\n"
        f"ASSISTANT RESPONSE:"
    )
    
    # Return standard OpenAI-compatible messaging block structure
    return [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

# Diagnostic validation trace
sample_q = "How often should I service my Swift?"
sample_ctx = [df['searchable_document'].iloc[0]] # Pulling the document from previous step data frame
structured_messages = build_rag_prompt(sample_q, sample_ctx)

print("Prompt Template compiled successfully. Structure breakdown:\n")
print(f"System Message Context:\n{structured_messages[0]['content']}\n")
print(f"User Message Context:\n{structured_messages[1]['content']}")

Prompt Template compiled successfully. Structure breakdown:

System Message Context:
You are an expert Automobile Workshop Customer Support Assistant.
Your goal is to provide accurate, professional, and clear answers to customer inquiries.

CRITICAL RULES:
1. Rely ONLY on the information provided in the 'RETIREVED WORKSHOP CONTEXT' block below.
2. Do NOT use outside general knowledge or guess answers.
3. If the provided context does not contain the answer to the user's question, respond exactly with:
   'I'm sorry, I couldn't find relevant information in our workshop database. Let me connect you with a service advisor.'
4. Keep your answers concise, clear, and actionable.

User Message Context:
RETIREVED WORKSHOP CONTEXT:
Vehicle: Maruti Swift
Category: FAQ
Problem: What is the recommended service interval?
Solution: Most vehicles recommend service every 10,000 km or 12 months, whichever comes first. Refer to the owner's manual for model-specific intervals and check severe usage schedu

# 🔄 Step 12: Build the RAG Pipeline

## Objective

Integrate retrieval and generation into a single workflow.

Pipeline:

```
User Query
      │
      ▼
Generate Query Embedding
      │
      ▼
Search ChromaDB
      │
      ▼
Retrieve Top-K Documents
      │
      ▼
Construct Prompt
      │
      ▼
Send to Cerebras LLM
      │
      ▼
Generate Response
      │
      ▼
Return Response
```

---

In [ ]:
# Running in-memory record of every turn in this session
conversation_history: List[Dict[str, str]] = []

def automobile_rag_pipeline(user_query: str, top_k: int = 3) -> str:
    """
    Executes the complete Retrieval-Augmented Generation pipeline.
    Retrieves semantic context, structures the prompt, queries Cerebras LLM,
    and stores the query/response pair in conversation_history.
    """
    try:
        # 1. Retrieve the top matching documents from ChromaDB
        retrieved_contexts = retrieve_relevant_context(user_query, top_k=top_k)

        # 2. Build the structured messages payload matching our prompt template
        messages_payload = build_rag_prompt(user_query, retrieved_contexts)

        # 3. Generate response via Cerebras using the configured MODEL
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages_payload,
            temperature=0.1,  # Keep temperature low for deterministic, grounded answers
            max_tokens=500
        )

        # 4. Extract the generated text
        bot_reply = response.choices[0].message.content.strip()

        # 5. Store this turn in the session's conversation history
        conversation_history.append({"query": user_query, "response": bot_reply})

        return bot_reply

    except Exception as e:
        error_message = f"An error occurred within the RAG pipeline execution: {str(e)}"
        conversation_history.append({"query": user_query, "response": error_message})
        return error_message

# Complete End-to-End Test
test_query = "What causes the engine to get too hot according to your files?"
print(f"Testing RAG Pipeline with Query: '{test_query}'\n")

pipeline_response = automobile_rag_pipeline(test_query)
print("=== Chatbot Grounded Response ===")
print(pipeline_response)

print("\n=== Stored Conversation History ===")
print(conversation_history)

Testing RAG Pipeline with Query: 'What causes the engine to get too hot according to your files?'

=== Chatbot Grounded Response ===
According to our records, an engine can overheat due to:

- Low coolant level  
- Blocked or leaking radiator  
- Cooling fan not operating correctly  
- Faulty thermostat  
- Damaged water pump or radiator hoses  

Checking these components is the first step in diagnosing the issue.


# 💾 Step 13: Save Conversation History

## Objective

Persist the session's conversation history (user queries and assistant responses) to a local JSON file.

This ensures that:

- Every customer query is permanently recorded, not just displayed and discarded
- Multiple chatbot sessions can be reviewed later for quality checks or audits
- Support staff can trace exactly what a customer asked and what the assistant answered

Each time the chatbot session ends, the current session's turns are appended to `chat_history.json`, preserving history from all previous sessions as well.

---

In [ ]:
import json
from datetime import datetime

def save_conversation_history(filepath: str = "chat_history.json") -> None:
    """
    Persists this session's conversation_history to a JSON file, appending
    to any previous sessions already saved there.
    """
    try:
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                existing = json.load(f)
        else:
            existing = []
    except (json.JSONDecodeError, FileNotFoundError):
        existing = []

    existing.append({
        "session_timestamp": datetime.now().isoformat(),
        "turns": conversation_history
    })

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2)

    print(f"💾 Conversation history saved to '{filepath}'.")

# 💬 Step 14: Create the Command Line Chatbot

## Objective

Implement an interactive chatbot that continuously accepts user queries.

For each query:

- Retrieve relevant documents
- Construct the prompt
- Generate a response
- Display the answer

The conversation continues until the user chooses to exit.

---

In [ ]:
import sys

def launch_workshop_chatbot():
    """
    Launches an interactive command-line interface loop for the 
    Automobile Workshop Support Assistant.
    """
    print("=================================================================")
    print("   🚙 Welcome to the Automobile Workshop Support Assistant! 🚙   ")
    print("=================================================================")
    print("Ask me anything about vehicle maintenance, troubleshooting, or schedules.")
    print("Type 'exit', 'quit', or 'bye' to close the session.\n")
    
    while True:
        try:
            # Capture input from the user and strip accidental trailing whitespace
            user_input = input("👤 You: ").strip()
            
            # Check for empty input strings to prevent empty vector search exceptions
            if not user_input:
                print("🤖 Assistant: Please type a valid question so I can query our records.\n")
                continue
                
            # Gracefully break out of the chat loop if an exit command is given
            if user_input.lower() in ['exit', 'quit', 'bye']:
                print("\n🤖 Assistant: Thank you for visiting our workshop assistant. Safe driving!")
                print("=================================================================")
                save_conversation_history()
                break
                
            # Process the user query through our end-to-end RAG pipeline
            print("🤖 Assistant: Checking our workshop database... 🔍")
            bot_response = automobile_rag_pipeline(user_input, top_k=3)
            
            # Display the final generated response back to the user
            print(f"\n🤖 Assistant: {bot_response}\n")
            print("-" * 50)  # Visual divider for readability between turns
            
        except KeyboardInterrupt:
            # Handle Ctrl+C interruptions elegantly without spilling tracebacks
            print("\n\n🤖 Assistant: Session closed unexpectedly. Have a wonderful day!")
            save_conversation_history()
            break
        except Exception as e:
            print(f"\n⚠️ Interface Error: An unexpected error occurred: {str(e)}\n")

# Run this line to start chatting directly in your terminal or Jupyter environment
launch_workshop_chatbot()

   🚙 Welcome to the Automobile Workshop Support Assistant! 🚙   
Ask me anything about vehicle maintenance, troubleshooting, or schedules.
Type 'exit', 'quit', or 'bye' to close the session.

🤖 Assistant: Checking our workshop database... 🔍

🤖 Assistant: Hello! How can I assist you today?

--------------------------------------------------
🤖 Assistant: Checking our workshop database... 🔍

🤖 Assistant: If your engine is overheating, follow these steps:

1. **Check the coolant level** – If it’s low, top it up with the correct coolant mixture.  
2. **Inspect the radiator** – Look for any blockages, debris, or leaks that could restrict airflow.  
3. **Verify the cooling fan operation** – Make sure the fan turns on when the engine reaches operating temperature.  
4. **Check the thermostat** – Ensure it opens at the proper temperature and isn’t stuck closed.  
5. **If the problem persists**, examine the water pump and radiator hoses for signs of damage or leaks.

If after these checks the ove

# ✅ Step 15: Test the Chatbot

## Objective

Evaluate the chatbot using realistic workshop-related questions.

Example Queries:

- My Tata Nexon engine is overheating.
- Why are my brakes making noise?
- How often should I change engine oil?
- My battery keeps draining overnight.
- Why is my car vibrating while idling?

Testing verifies that retrieval and generation work together correctly.

---

In [40]:
# A structured list containing your evaluation questions
test_scenarios = [
    "My Tata Nexon engine is overheating.",
    "Why are my brakes making noise?",
    "How often should I change engine oil?",
    "My battery keeps draining overnight.",
    "Why is my car vibrating while idling?",
    "Can you give me the replacement steps for a SpaceX Falcon 9 engine valve?" # Out-of-bounds safety check
]

print("=================================================================")
print("🤖 RUNNING AUTOMATED RAG PIPELINE BATCH EVALUATION SEED TEST 🤖")
print("=================================================================\n")

for i, scenario in enumerate(test_scenarios, 1):
    print(f"📋 TEST SCENARIO #{i}")
    print(f"👤 Query: '{scenario}'")
    print("🔍 Pipeline Action: Embedding query -> Searching ChromaDB -> Synthesizing context...")
    
    # Process the evaluation question through our end-to-end RAG pipeline
    reply = automobile_rag_pipeline(scenario, top_k=2)
    
    print("🤖 Assistant Response:")
    print(f"{reply}")
    print("\n" + "="*65 + "\n")

print("Evaluation run completed successfully.")

🤖 RUNNING AUTOMATED RAG PIPELINE BATCH EVALUATION SEED TEST 🤖

📋 TEST SCENARIO #1
👤 Query: 'My Tata Nexon engine is overheating.'
🔍 Pipeline Action: Embedding query -> Searching ChromaDB -> Synthesizing context...
🤖 Assistant Response:
For a Tata Nexon that’s overheating, follow these steps:

1. **Coolant level** – Verify the coolant is at the proper level; top it up if it’s low.  
2. **Radiator** – Look for any blockages or leaks in the radiator and clear or repair them.  
3. **Cooling fan** – Ensure the fan turns on and runs correctly when the engine reaches operating temperature.  
4. **Thermostat** – Check that the thermostat opens at the right temperature and isn’t stuck closed.  

If the temperature is still high after these checks, inspect the **water pump** and **radiator hoses** for any damage or leaks and replace them as needed.


📋 TEST SCENARIO #2
👤 Query: 'Why are my brakes making noise?'
🔍 Pipeline Action: Embedding query -> Searching ChromaDB -> Synthesizing context...
🤖

# 📖 Conclusion

This project demonstrates the implementation of a simple Retrieval-Augmented Generation (RAG) system for automobile workshop customer support.

By combining semantic search using ChromaDB with the reasoning capabilities of the Cerebras LLM, the chatbot provides responses that are more accurate, relevant, and grounded in the provided knowledge base than a standalone LLM.